# Biomedical NER + Biomedical Entity Linking Pipeline

The workflow has two main stages:

1. **Named Entity Recognition (NER)** using GLiNER.
2. **Biomedical Entity Linking / Normalisation (BEL)** using a mixed strategy:
   - rule-based normalization for structured entities;
   - ontology-based linking for biomedical concepts;
   - LLM-assisted classification for contextual entities.

## 1. Environment setup

In [ ]:
!pip install -q gliner torch transformers accelerate faiss-cpu

In [ ]:
import json
import re
from pathlib import Path
from typing import List, Dict, Any, Optional
from collections import Counter, defaultdict

import torch
from gliner import GLiNER
from transformers import AutoTokenizer, AutoModel

try:
    from google.colab import files
except Exception:
    files = None

## 2. GLiNER BioMed extraction

In [ ]:
OUT_DIR = Path("./gliner_results")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ID = "Ihor/gliner-biomed-large-v1.0"
GLOBAL_THRESHOLD = 0.70

LABEL_THRESHOLDS = {
    "dose": 0.65,
    "frequency_or_interval": 0.65,
    "eligibility_criterion": 0.65,
}

CHUNK_MAX_TOKENS = 512
CHUNK_OVERLAP_TOKENS = 50
PARAGRAPHS_PER_CHUNK = 4
PARAGRAPH_OVERLAP = 1

device = "cuda" if torch.cuda.is_available() else "cpu"
model = GLiNER.from_pretrained(MODEL_ID).to(device)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
print("Model loaded on", device)

In [ ]:
LABELS = [
    "population",
    "condition",
    "anatomy",
    "stage_or_risk",
    "eligibility_criterion",
    "histology_or_subtype",
    "biomarker_or_gene",
    "diagnostic_test",
    "intervention",
    "drug",
    "regimen",
    "dose",
    "route",
    "frequency_or_interval",
    "duration_or_timing",
    "contraindication_or_caution",
    "adverse_event",
    "evidence_or_category",
    "recommendation_strength",
    "outcome_or_goal",
]

STRUCTURAL_LABELS = list(LABEL_THRESHOLDS.keys())
CONCEPT_LABELS = [label for label in LABELS if label not in STRUCTURAL_LABELS]

print(f"Labels: {len(LABELS)} total")
print("Structural labels:", STRUCTURAL_LABELS)

### 2.1 Text cleaning

The cleaning stage removes common Markdown/PDF conversion artefacts while preserving clinical expressions such as doses, comparison signs, biomarker abbreviations and short medical forms.

In [ ]:
def _strip_document_header(text: str) -> str:
    markers = [
        r"^## Summary of the Guidelines Updates",
        r"^## Updates in Version",
        r"^## Overview",
        r"^## Principles of",
        r"^## Emetogenic",
        r"^## HIGH ",
        r"^## MODERATE ",
        r"^## LOW ",
    ]
    for marker in markers:
        match = re.search(marker, text, flags=re.MULTILINE | re.IGNORECASE)
        if match:
            return text[match.start():]
    return text


def _fix_mathbf(match: re.Match) -> str:
    value = match.group(1).replace("~", " ")
    value = re.sub(r"\s*([\-\.,/])\s*", r"\1", value)
    value = re.sub(r"([0-9])\s+([0-9])", r"\1\2", value)
    value = re.sub(r"([><=])\s+", r"\1", value)
    value = re.sub(r"([a-zA-Z])\s+([a-zA-Z])", r"\1\2", value)
    value = re.sub(r"(\d)([a-zA-Z])", r"\1 \2", value)
    return value.strip()


def _process_latex_block(match: re.Match) -> str:
    value = match.group(1).strip()
    value = re.sub(r"\\boldsymbol\{\\geq\}", "≥", value)
    value = re.sub(r"\\boldsymbol\{\\leq\}", "≤", value)
    value = re.sub(r"\\boldsymbol\{([^}]*)\}", r"\1", value)
    value = re.sub(r"\\mathbf\{([^}]+)\}", _fix_mathbf, value)
    value = re.sub(r"\\mathrm\{([^}]+)\}", r"\1", value)
    value = re.sub(r"\\text(?:bf|it|rm)?\{([^}]+)\}", r"\1", value)
    value = value.replace(r"\geq", "≥").replace(r"\leq", "≤")
    value = value.replace("~", " ")
    value = re.sub(r"\s+", " ", value).strip()
    return value


def clean_guideline_text(text: str, strip_header: bool = True) -> str:
    text = text or ""
    if strip_header:
        text = _strip_document_header(text)

    text = re.sub(r"!\[.*?\]\(https?://[^\)]+\)", " ", text, flags=re.DOTALL)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\$([^$\n]{1,160})\$", _process_latex_block, text)
    text = re.sub(r"\{#.*?\}", " ", text)
    text = re.sub(r"\[(?:Back to Top|Table of Contents)\]", " ", text, flags=re.I)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]{2,}", " ", text)
    return text.strip()

### 2.2 Paragraph packing and token-safe splitting

The first split keeps neighbouring clinical statements together. The second split prevents the model from receiving overlong sequences.

In [ ]:
def paragraph_pack(text: str, max_paras: int = PARAGRAPHS_PER_CHUNK, overlap_paras: int = PARAGRAPH_OVERLAP) -> List[str]:
    paragraphs = [p.strip() for p in text.split("

") if p.strip()]
    groups: List[str] = []
    i = 0

    while i < len(paragraphs):
        group = "

".join(paragraphs[i:i + max_paras])
        if group.strip():
            groups.append(group)
        if i + max_paras >= len(paragraphs):
            break
        i = max(0, i + max_paras - overlap_paras)

    return groups


def token_safe_split(text: str, max_tokens: int = CHUNK_MAX_TOKENS, overlap_tokens: int = CHUNK_OVERLAP_TOKENS) -> List[str]:
    encoded = tokenizer(text, return_offsets_mapping=True, add_special_tokens=False)
    offsets = encoded["offset_mapping"]

    if len(offsets) <= max_tokens:
        return [text]

    pieces: List[str] = []
    start = 0
    while start < len(offsets):
        end = min(start + max_tokens, len(offsets))
        piece = text[offsets[start][0]:offsets[end - 1][1]].strip()
        if piece:
            pieces.append(piece)
        if end == len(offsets):
            break
        start = max(0, end - overlap_tokens)

    return pieces


def build_chunks(raw_text: str) -> List[Dict[str, Any]]:
    cleaned = clean_guideline_text(raw_text)
    chunks: List[Dict[str, Any]] = []

    for group_id, paragraph_group in enumerate(paragraph_pack(cleaned)):
        for split_id, chunk_text in enumerate(token_safe_split(paragraph_group)):
            chunks.append({
                "group_id": group_id,
                "split_id": split_id,
                "text": chunk_text,
                "n_chars": len(chunk_text),
                "n_tokens": len(tokenizer(chunk_text, add_special_tokens=False)["input_ids"]),
            })

    return chunks

### 2.3 Post-filtering policy

The policy removes obvious conversion artefacts, generic placeholders and several common label confusions before the entities are passed to the linking stage.

In [ ]:
STOPWORDS = {
    "et", "al", "et al", "eg", "ie", "etc",
    "fig", "figure", "table", "tab", "appendix",
    "na", "n/a", "nr", "unknown",
}

META_DROP_RX = re.compile(
    r"(guidelines?\s+version|nccn\s+guidelines?|version\s+\d+(\.\d+)*|copyright|©)",
    re.I,
)

POP_GENERIC = {
    "patient", "patients", "person", "persons", "people", "individual", "individuals",
    "population", "group", "groups", "cohort", "cohorts",
}

COND_GENERIC = {
    "disease", "cancer", "carcinoma", "tumor", "tumour", "malignancy",
    "infection", "disorder", "syndrome", "finding", "findings",
}

STATUS_RX = re.compile(
    r"\b(metastatic|recurrent|relapsed|refractory|advanced|locally\s+advanced|"
    r"unresectable|inoperable|stage\s*[0-4ivx]+|grade\s*[0-4ivx]+|"
    r"[pc]?[TNM]\d[a-c]?)\b",
    re.I,
)

AE_RX = re.compile(
    r"\b(neutropen(ia|ic)|thrombocytopen(ia|ic)|anemi(a|c)|pneumonitis|"
    r"colitis|hepatitis|infection|sepsis|fever|fatigue|rash|neuropathy|"
    r"nausea|vomit|diarrh(o)?ea|AKI|TLS|ICANS|\w+emia\b|\w+penia\b)\b",
    re.I,
)

DOSE_UNIT_RX = re.compile(r"\b(mg|mcg|µg|g|iu|units?|mg/kg|mcg/kg|gy|cgy|auc)\b", re.I)
FREQ_KEEP_RX = re.compile(r"\b(q\d+h|q\d+w|qod|qd|bid|tid|qid|prn|once|twice|daily|weekly|monthly|every)\b", re.I)
DUR_KEEP_RX = re.compile(r"\b(day|days|week|weeks|month|months|year|years|cycle|cycles|within|before|after|post|pre)\b", re.I)
ROUTE_KEEP_RX = re.compile(r"\b(oral|po\b|iv\b|intravenous|infusion|subcutaneous|sc\b|intramuscular|im\b|topical|inhaled)\b", re.I)


def normalize_entity_text(text: str) -> str:
    text = re.sub(r"\s+", " ", (text or "").strip())
    return re.sub(r"^[\W_]+|[\W_]+$", "", text)


def route_entity(entity: Dict[str, Any], new_label: str, text: str, score: float) -> Dict[str, Any]:
    routed = dict(entity)
    routed["label"] = new_label
    routed["text"] = text
    routed["score"] = score
    routed["routed"] = True
    return routed


def apply_policy(entity: Dict[str, Any]) -> List[Dict[str, Any]]:
    label = (entity.get("label") or "").strip()
    text = normalize_entity_text(entity.get("text") or "")
    score = float(entity.get("score", 0.0))

    if not label or not text:
        return []

    low = text.lower()
    if low in STOPWORDS or META_DROP_RX.search(text):
        return []

    if label == "population" and low in POP_GENERIC:
        return []

    if label == "condition":
        if low in COND_GENERIC:
            return []
        if STATUS_RX.search(text) and not AE_RX.search(text):
            return [route_entity(entity, "stage_or_risk", text, score)]
        if AE_RX.search(text) and not STATUS_RX.search(text):
            return [route_entity(entity, "adverse_event", text, score)]

    if label == "dose" and not DOSE_UNIT_RX.search(text):
        return []

    if label == "frequency_or_interval" and not FREQ_KEEP_RX.search(text):
        return []

    if label == "duration_or_timing" and not DUR_KEEP_RX.search(text):
        return []

    if label == "route" and not ROUTE_KEEP_RX.search(text):
        return []

    if label in LABEL_THRESHOLDS and score < LABEL_THRESHOLDS[label]:
        return []

    cleaned = dict(entity)
    cleaned["text"] = text
    cleaned["score"] = score
    cleaned.setdefault("routed", False)
    return [cleaned]

### 2.4 Document-level extraction

The model is called twice per chunk: once for general biomedical concepts and once for structurally sensitive labels with their own threshold policy.

In [ ]:
def predict_chunk_entities(chunk_text: str, chunk_id: int, doc_id: str) -> List[Dict[str, Any]]:
    predictions = []
    predictions.extend(model.predict_entities(chunk_text, CONCEPT_LABELS, threshold=GLOBAL_THRESHOLD))
    predictions.extend(model.predict_entities(chunk_text, STRUCTURAL_LABELS, threshold=min(LABEL_THRESHOLDS.values())))

    records: List[Dict[str, Any]] = []
    for pred in predictions:
        record = {
            "doc_id": doc_id,
            "chunk_id": chunk_id,
            "label": pred.get("label"),
            "text": pred.get("text"),
            "score": float(pred.get("score", 0.0)),
            "start": pred.get("start"),
            "end": pred.get("end"),
        }
        records.extend(apply_policy(record))

    return records


def deduplicate_entities(records: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    seen = set()
    result = []

    for record in sorted(records, key=lambda x: (-float(x.get("score", 0.0)), x.get("label", ""), x.get("text", ""))):
        key = (
            record.get("doc_id"),
            record.get("chunk_id"),
            record.get("label"),
            normalize_entity_text(record.get("text", "")).lower(),
        )
        if key in seen:
            continue
        seen.add(key)
        result.append(record)

    return result


def extract_entities_from_document(raw_md: str, doc_id: str) -> List[Dict[str, Any]]:
    chunks = build_chunks(raw_md)
    collected: List[Dict[str, Any]] = []

    for chunk_id, chunk in enumerate(chunks):
        chunk_records = predict_chunk_entities(chunk["text"], chunk_id=chunk_id, doc_id=doc_id)
        for record in chunk_records:
            record["n_tokens"] = chunk["n_tokens"]
        collected.extend(chunk_records)

    return deduplicate_entities(collected)

### 2.5 Batch processing and export

The same extraction function can be applied either to one Markdown document or to a folder/ZIP archive of guideline files.

In [ ]:
def run_gliner_on_folder(md_dir: Path, out_dir: Path = OUT_DIR) -> Dict[str, List[Dict[str, Any]]]:
    out_dir.mkdir(parents=True, exist_ok=True)
    combined: Dict[str, List[Dict[str, Any]]] = {}

    for path in sorted(md_dir.rglob("*.md")):
        doc_id = path.stem
        raw_text = path.read_text(encoding="utf-8", errors="ignore")
        entities = extract_entities_from_document(raw_text, doc_id=doc_id)
        combined[doc_id] = entities

        with (out_dir / f"gliner_summary_{doc_id}.json").open("w", encoding="utf-8") as f:
            json.dump({doc_id: entities}, f, ensure_ascii=False, indent=2)

        counts = Counter(entity["label"] for entity in entities)
        print(f"{doc_id}: {len(entities)} entities | {dict(counts.most_common(5))}")

    with (out_dir / "gliner_combined.json").open("w", encoding="utf-8") as f:
        json.dump(combined, f, ensure_ascii=False, indent=2)

    return combined


def summarize_gliner_output(combined: Dict[str, List[Dict[str, Any]]]) -> Dict[str, Any]:
    label_counts = Counter()
    doc_counts = {}

    for doc_id, records in combined.items():
        doc_counts[doc_id] = len(records)
        label_counts.update(record.get("label") for record in records)

    return {
        "n_documents": len(combined),
        "n_entities": sum(doc_counts.values()),
        "top_labels": label_counts.most_common(20),
        "documents": doc_counts,
    }

In [ ]:
sample_md = """
## Overview
Patients with metastatic non-small cell lung cancer may receive pembrolizumab 200 mg IV every 3 weeks.
Patients with severe pneumonitis should discontinue therapy.
"""

sample_entities = extract_entities_from_document(sample_md, doc_id="sample_guideline")
sample_entities[:10]

In [ ]:
sample_combined = {"sample_guideline": sample_entities}
summarize_gliner_output(sample_combined)

## 3. Entity grouping for BEL

In [ ]:
GROUP_B = ["dose", "frequency_or_interval", "route", "duration_or_timing"]
GROUP_C = ["recommendation_strength", "evidence_or_category", "stage_or_risk", "regimen"]
GROUP_D = [
    "eligibility_criterion", "population", "contraindication_or_caution",
    "intervention", "outcome_or_goal",
]
GROUP_A = [
    "drug", "condition", "biomarker_or_gene", "histology_or_subtype",
    "diagnostic_test", "anatomy", "adverse_event",
]

ALL_GROUPS = {"A": GROUP_A, "B": GROUP_B, "C": GROUP_C, "D": GROUP_D}

In [ ]:
def summarize_unique_values(data: Dict[str, List[Dict[str, Any]]]) -> Dict[str, Any]:
    all_entities = [entity for entities in data.values() for entity in entities]

    result = {}
    for group_name, labels in ALL_GROUPS.items():
        result[group_name] = {}
        for label in labels:
            counts = Counter(
                entity["text"].strip()
                for entity in all_entities
                if entity.get("label") == label
            )
            result[group_name][label] = {
                "total": sum(counts.values()),
                "unique": len(counts),
                "values": [[text, n] for text, n in counts.most_common(20)],
            }
    return result

## 4. Group B — rule-based normalisation

In [ ]:
ALL_NORMALIZERS = {}

ROUTE_LOOKUP = {
    "iv": "IV",
    "i.v.": "IV",
    "intravenous": "IV",
    "intravenously": "IV",
    "po": "PO",
    "p.o.": "PO",
    "oral": "PO",
    "orally": "PO",
    "by mouth": "PO",
    "subcutaneous": "SC",
    "subcutaneously": "SC",
    "sc": "SC",
    "s.c.": "SC",
    "intramuscular": "IM",
    "im": "IM",
    "topical": "TOPICAL",
    "inhaled": "INHALATION",
}


def normalize_route(text: str) -> Optional[Dict[str, Any]]:
    key = re.sub(r"\s+", " ", text.strip().lower())
    route = ROUTE_LOOKUP.get(key)
    if route:
        return {"normalized_route": route}
    return None


FREQ_LOOKUP = {
    "daily": "QD",
    "once daily": "QD",
    "once a day": "QD",
    "qd": "QD",
    "twice daily": "BID",
    "twice a day": "BID",
    "bid": "BID",
    "three times daily": "TID",
    "tid": "TID",
    "four times daily": "QID",
    "qid": "QID",
    "weekly": "QW",
    "once weekly": "QW",
    "every week": "QW",
    "every 2 weeks": "Q2W",
    "every 3 weeks": "Q3W",
    "every 4 weeks": "Q4W",
}


def normalize_frequency(text: str) -> Optional[Dict[str, Any]]:
    key = re.sub(r"\s+", " ", text.strip().lower())
    if key in FREQ_LOOKUP:
        return {"normalized_frequency": FREQ_LOOKUP[key]}

    m = re.search(r"every\s+(\d+)\s+(hour|hours|day|days|week|weeks|month|months)", key)
    if m:
        value, unit = int(m.group(1)), m.group(2)
        unit_code = {
            "hour": "H", "hours": "H",
            "day": "D", "days": "D",
            "week": "W", "weeks": "W",
            "month": "M", "months": "M",
        }[unit]
        return {"normalized_frequency": f"Q{value}{unit_code}"}

    m = re.search(r"(\d+)\s*times?\s+(?:per|a)\s+(day|week|month)", key)
    if m:
        value, unit = int(m.group(1)), m.group(2)
        return {"frequency_count": value, "frequency_period": unit}

    return None


In [ ]:
UCUM_UNITS = {
    "mg": "mg",
    "g": "g",
    "mcg": "ug",
    "µg": "ug",
    "ug": "ug",
    "ml": "mL",
    "iu": "[IU]",
    "units": "[IU]",
}


def normalize_dose(text: str) -> Optional[Dict[str, Any]]:
    key = re.sub(r"\s+", " ", text.strip().lower())

    m = re.search(r"(\d+(?:\.\d+)?)\s*(mg|g|mcg|µg|ug|ml|iu|units)\b", key)
    if m:
        return {
            "dose_value": float(m.group(1)),
            "dose_unit": UCUM_UNITS[m.group(2)],
        }

    m = re.search(r"(\d+(?:\.\d+)?)\s*(mg|g|mcg|µg|ug)\s*/\s*(kg|m2|m\^2)", key)
    if m:
        denominator = {"kg": "kg", "m2": "m2", "m^2": "m2"}[m.group(3)]
        return {
            "dose_value": float(m.group(1)),
            "dose_unit": UCUM_UNITS[m.group(2)],
            "dose_per": denominator,
        }

    m = re.search(r"(\d+(?:\.\d+)?)\s*%", key)
    if m:
        return {"dose_value": float(m.group(1)), "dose_unit": "%"}

    return None


DURATION_UNITS = {
    "day": "days", "days": "days",
    "week": "weeks", "weeks": "weeks",
    "cycle": "cycles", "cycles": "cycles",
    "month": "months", "months": "months",
    "year": "years", "years": "years",
}

TIMING_LOOKUP = {
    "until disease progression": {"timing_type": "until_event", "event": "disease_progression"},
    "until progression": {"timing_type": "until_event", "event": "disease_progression"},
    "until unacceptable toxicity": {"timing_type": "until_event", "event": "unacceptable_toxicity"},
    "until toxicity": {"timing_type": "until_event", "event": "toxicity"},
    "as clinically indicated": {"timing_type": "clinical_judgement"},
}


def normalize_duration(text: str) -> Optional[Dict[str, Any]]:
    key = re.sub(r"\s+", " ", text.strip().lower())

    if key in TIMING_LOOKUP:
        return TIMING_LOOKUP[key]

    m = re.search(r"(?:for|during)?\s*(\d+)\s+(day|days|week|weeks|cycle|cycles|month|months|year|years)", key)
    if m:
        return {
            "duration_value": int(m.group(1)),
            "duration_unit": DURATION_UNITS[m.group(2)],
        }

    m = re.search(r"cycles?\s+(\d+)\s*[-–]\s*(\d+)", key)
    if m:
        return {
            "timing_type": "cycle_range",
            "cycle_start": int(m.group(1)),
            "cycle_end": int(m.group(2)),
        }

    return None


ALL_NORMALIZERS.update({
    "route": normalize_route,
    "frequency_or_interval": normalize_frequency,
    "dose": normalize_dose,
    "duration_or_timing": normalize_duration,
})


In [ ]:
group_b_examples = [
    {"label": "route", "text": "intravenously"},
    {"label": "route", "text": "by mouth"},
    {"label": "dose", "text": "2 mg/kg"},
    {"label": "dose", "text": "0.5 mg"},
    {"label": "frequency_or_interval", "text": "twice daily"},
    {"label": "frequency_or_interval", "text": "every 4 weeks"},
    {"label": "duration_or_timing", "text": "for 6 cycles"},
    {"label": "duration_or_timing", "text": "until disease progression"},
]

[
    {
        "label": item["label"],
        "text": item["text"],
        "normalized": ALL_NORMALIZERS[item["label"]](item["text"]),
    }
    for item in group_b_examples
]


## 5. Group C — semi-structured clinical attributes

In [ ]:
EVIDENCE_LOOKUP = {
    "category 1": {"nccn_category": "1", "evidence_level": "high_level"},
    "category 2a": {"nccn_category": "2A", "evidence_level": "lower_level"},
    "category 2b": {"nccn_category": "2B", "evidence_level": "lower_level"},
}


def normalize_evidence(text: str) -> Optional[Dict[str, Any]]:
    return EVIDENCE_LOOKUP.get(text.strip().lower())


STRENGTH_LOOKUP = {
    "preferred": {"strength_level": "preferred"},
    "recommended": {"strength_level": "recommended"},
    "useful in certain circumstances": {"strength_level": "conditional"},
}


def normalize_strength(text: str) -> Optional[Dict[str, Any]]:
    return STRENGTH_LOOKUP.get(text.strip().lower())


ROMAN_TO_STAGE = {
    "0": "0", "i": "I", "ii": "II", "iii": "III", "iv": "IV",
    "1": "I", "2": "II", "3": "III", "4": "IV",
}


def normalize_stage(text: str) -> Optional[Dict[str, Any]]:
    key = text.strip().lower()
    m = re.search(r"stage\s+([0ivx]+)", key)
    if m:
        return {"stage": ROMAN_TO_STAGE.get(m.group(1), m.group(1).upper())}

    if "metastatic" in key:
        return {"disease_status": "metastatic"}
    if "recurrent" in key:
        return {"disease_status": "recurrent"}

    return None

In [ ]:
REGIMEN_ACRONYMS = {
    "folfox": ("FOLFOX", "chemotherapy"),
    "folfiri": ("FOLFIRI", "chemotherapy"),
    "capeox": ("CAPEOX", "chemotherapy"),
    "r-chop": ("R-CHOP", "chemoimmunotherapy"),
}


def normalize_regimen(text: str) -> Optional[Dict[str, Any]]:
    key = text.strip().lower()
    if key in REGIMEN_ACRONYMS:
        regimen_name, regimen_type = REGIMEN_ACRONYMS[key]
        return {
            "regimen_name": regimen_name,
            "regimen_type": regimen_type,
        }
    return None


ALL_NORMALIZERS.update({
    "evidence_or_category": normalize_evidence,
    "recommendation_strength": normalize_strength,
    "stage_or_risk": normalize_stage,
    "regimen": normalize_regimen,
})

## 6. Group D — contextual entities

In [ ]:
AGE_OP = {
    ">": "older_than", ">=": "at_least", "≥": "at_least",
    "<": "younger_than", "<=": "at_most", "≤": "at_most",
}


def normalize_population(text: str) -> Optional[Dict[str, Any]]:
    key = text.strip().lower()
    result = {}

    if re.search(r"\b(women|female|females)\b", key):
        result["sex"] = "female"
    elif re.search(r"\b(men|male|males)\b", key):
        result["sex"] = "male"

    m = re.search(r"([<>≥≤]=?)\s*(\d+)\s*(?:years?|yrs?)", key)
    if m:
        result["age_operator"] = AGE_OP.get(m.group(1), m.group(1))
        result["age_value"] = int(m.group(2))

    return result or None


OUTCOME_LOOKUP = {
    "os": ("overall_survival", "survival"),
    "pfs": ("progression_free_survival", "survival"),
    "orr": ("overall_response_rate", "response"),
    "cr": ("complete_response", "response"),
}


def normalize_outcome(text: str) -> Optional[Dict[str, Any]]:
    key = text.strip().lower()
    if key in OUTCOME_LOOKUP:
        name, family = OUTCOME_LOOKUP[key]
        return {"outcome_name": name, "outcome_family": family}
    return None

In [ ]:
CONTRA_LOOKUP = {
    "contraindicated": "absolute",
    "contraindication": "absolute",
    "not recommended": "relative",
    "caution": "relative",
}


def normalize_contraindication(text: str) -> Optional[Dict[str, Any]]:
    key = text.strip().lower()
    for phrase, level in CONTRA_LOOKUP.items():
        if phrase in key:
            return {"caution_level": level}
    return None


def normalize_eligibility(text: str) -> Optional[Dict[str, Any]]:
    key = text.strip().lower()

    m = re.search(r"(?:patients?|adults?|individuals?)\s*(?:aged?\s*)?([<>≥≤]=?)\s*(\d+)", key)
    if m:
        return {
            "criterion_type": "age",
            "age_operator": AGE_OP.get(m.group(1), m.group(1)),
            "age_value": int(m.group(2)),
        }

    if "ecog" in key:
        return {"criterion_type": "performance_status"}

    return None


def normalize_intervention(text: str) -> Optional[Dict[str, Any]]:
    key = text.strip().lower()

    if re.search(r"\bradiation\b|\bradiotherapy\b|\bsbrt\b|\bimrt\b", key):
        return {"intervention_type": "radiation"}
    if re.search(r"\bsurgery\b|\bresection\b|\bsurgical\b", key):
        return {"intervention_type": "surgery"}
    if re.search(r"\bchemotherapy\b|\bchemo\b", key):
        return {"intervention_type": "chemotherapy"}

    return None


ALL_NORMALIZERS.update({
    "population": normalize_population,
    "outcome_or_goal": normalize_outcome,
    "contraindication_or_caution": normalize_contraindication,
    "eligibility_criterion": normalize_eligibility,
    "intervention": normalize_intervention,
})

## 7. LLM-assisted classification for unresolved contextual entities

In [ ]:
PROMPT_TEMPLATE = """
You are a biomedical information extraction assistant.
Classify the entity into one of the predefined biomedical categories.
Return a compact JSON object with the selected category and a short rationale.

Entity: {entity}
Original label: {label}
Context: {context}
""".strip()


def build_classification_prompt(entity: Dict[str, Any], context: str = "") -> str:
    return PROMPT_TEMPLATE.format(
        entity=entity.get("text", ""),
        label=entity.get("label", ""),
        context=context[:800],
    )

## 8. Group A — ontology-based biomedical entity linking

In [ ]:
SAPBERT_MODEL_ID = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"

# sapbert_tokenizer = AutoTokenizer.from_pretrained(SAPBERT_MODEL_ID)
# sapbert_model = AutoModel.from_pretrained(SAPBERT_MODEL_ID)


In [ ]:
def link_entity_with_ontology(entity: Dict[str, Any]) -> Dict[str, Any]:
    label = entity.get("label")
    text = entity.get("text", "")

    return {
        "text": text,
        "label": label,
        "linked": False,
        "el_method": "ontology_candidate_retrieval",
        "candidate_term": None,
        "candidate_id": None,
        "score": None,
    }


example_link = link_entity_with_ontology({"text": "aspirin", "label": "drug"})
example_link

## 9. Run normalisation / linking

In [ ]:
def normalize_record(entity: Dict[str, Any]) -> Dict[str, Any]:
    label = entity.get("label")
    text = entity.get("text", "")

    record = {
        "doc_id": entity.get("doc_id"),
        "chunk_id": entity.get("chunk_id"),
        "label": label,
        "text": text,
        "ner_score": entity.get("score"),
    }

    if label in GROUP_A:
        record.update(link_entity_with_ontology(entity))
        return record

    normalizer = ALL_NORMALIZERS.get(label)
    if normalizer is None:
        record.update({"linked": False, "el_method": "not_applicable"})
        return record

    normalized = normalizer(text)
    record.update({
        "linked": normalized is not None,
        "el_method": "rule_based",
    })
    if normalized:
        record.update(normalized)

    return record


def run_bel_pipeline(data: Dict[str, List[Dict[str, Any]]]) -> List[Dict[str, Any]]:
    records = []
    for entities in data.values():
        for entity in entities:
            records.append(normalize_record(entity))
    return records

In [ ]:
demo_data = {
    "sample_guideline": [
        {"doc_id": "sample_guideline", "chunk_id": 0, "label": "drug", "text": "aspirin", "score": 0.91},
        {"doc_id": "sample_guideline", "chunk_id": 0, "label": "dose", "text": "200 mg", "score": 0.88},
        {"doc_id": "sample_guideline", "chunk_id": 0, "label": "dose", "text": "2 mg/kg", "score": 0.86},
        {"doc_id": "sample_guideline", "chunk_id": 0, "label": "route", "text": "IV", "score": 0.83},
        {"doc_id": "sample_guideline", "chunk_id": 0, "label": "frequency_or_interval", "text": "every 3 weeks", "score": 0.82},
        {"doc_id": "sample_guideline", "chunk_id": 0, "label": "duration_or_timing", "text": "until disease progression", "score": 0.79},
        {"doc_id": "sample_guideline", "chunk_id": 0, "label": "stage_or_risk", "text": "metastatic disease", "score": 0.80},
    ]
}

bel_records = run_bel_pipeline(demo_data)
bel_records


## 10. Consolidated output

In [ ]:
def consolidate_records(records: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    consolidated = []
    seen = set()

    for record in records:
        key = (
            record.get("doc_id"),
            record.get("chunk_id"),
            record.get("label"),
            str(record.get("text", "")).lower(),
        )
        if key in seen:
            continue
        seen.add(key)
        consolidated.append(record)

    return consolidated


consolidated = consolidate_records(bel_records)
consolidated

In [ ]:
OUTPUT_FILE = "entities_consolidated_demo.json"

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(consolidated, f, ensure_ascii=False, indent=2)

print(f"Saved {len(consolidated)} records to {OUTPUT_FILE}")